# Spicerack: A Recipe Recommendation System Based on Users Spices


## Background
A common problem when shopping for groceries is forgetting what spices you already have at home. Spicerack helps you understand which spices you own, how they form flavor profiles, and what recipes you can cook — plus what single spice to buy next to unlock the most new options.


---
## STEP 1 — SET YOUR INPUTS HERE
**This is the only cell you need to edit.** Change your pantry and must-use spices, then run all cells top to bottom.

Valid spices you can choose from:
`salt, black pepper, garlic, garlic powder, onion powder, cumin, paprika, smoked paprika, chili powder, crushed red pepper, cayenne, turmeric, coriander, ginger, cinnamon, nutmeg, cloves, allspice, oregano, basil, thyme, rosemary, sage, bay leaf, parsley, dill, tarragon, cardamom, fennel, mustard, mustard powder, curry powder, garam masala, star anise, anise, sumac, za'atar`

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
#  EDIT THESE TWO LISTS — everything else runs automatically
# ─────────────────────────────────────────────────────────────────────────────

# Spices you currently have in your pantry
MY_PANTRY = [
    "garlic",
    "cumin",
    "paprika",
    "chili powder",
    "oregano",
    "black pepper",
    "salt",
    "cinnamon",
    "ginger",
    "sumac",
]

# Spices you WANT TO USE — these MUST appear in every recommended recipe
# Set to [] if you have no requirements
MUST_USE = [
    "cumin",
    "garlic",
    "sumac",
]

# Path to your RecipeNLG CSV file
CSV_PATH = "/Users/daniellarson/Desktop/SpiceRack/cookingdataset/RecipeNLG_dataset.csv"

# How many recipes to load (reduce if your laptop is slow, increase for better results)
SAMPLE_SIZE = 100_000

---
## Setup — Imports, Vocabulary & Flavor Profiles
*Run once. Do not edit.*

In [31]:
import re
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import pairwise_distances

In [32]:
# ── Hardlocked spice vocabulary ───────────────────────────────────────────────
SPICES = [
    "salt", "black pepper", "garlic", "garlic powder",
    "onion powder",
    "cumin", "paprika", "smoked paprika",
    "chili powder", "crushed red pepper", "cayenne",
    "turmeric", "coriander",
    "ginger", "cinnamon", "nutmeg", "cloves", "allspice",
    "oregano", "basil", "thyme", "rosemary", "sage",
    "bay leaf", "parsley", "dill", "tarragon",
    "cardamom", "fennel", "mustard", "mustard powder",
    "curry powder", "garam masala",
    "star anise", "anise",
    "sumac", "za'atar",
]

# Aliases: variant names → canonical name
ALIASES = {
    "bay leaves":        "bay leaf",
    "red pepper flakes": "crushed red pepper",
    "ground ginger":     "ginger",
    "garlic powder":     "garlic",
    "smoked paprika":    "paprika",
    "pepper":            "black pepper",
    "mustard powder":    "mustard",
}

CANONICAL_SPICES = sorted(set(ALIASES.get(s, s) for s in SPICES))

# ── Flavor profiles ───────────────────────────────────────────────────────────
FLAVOR_PROFILES = {
    "Smoky & Savory":           {"paprika", "cumin", "chili powder", "garlic", "black pepper", "cayenne"},
    "Warm & Sweet":             {"cinnamon", "nutmeg", "cloves", "allspice", "ginger", "cardamom", "star anise"},
    "Herby & Mediterranean":    {"oregano", "basil", "thyme", "rosemary", "sage", "bay leaf", "parsley", "tarragon", "dill"},
    "Bright & Tangy":           {"sumac", "za'atar", "coriander", "fennel", "mustard", "turmeric"},
    "South Asian & Aromatic":   {"curry powder", "garam masala", "turmeric", "coriander", "cumin", "ginger", "cardamom"},
    "Bold & Spicy":             {"cayenne", "crushed red pepper", "chili powder", "black pepper", "cumin", "garlic"},
}

# ── Culinary regions ──────────────────────────────────────────────────────────
REGION_PROFILES = {
    "Mexican / Tex-Mex":        ["Smoky & Savory", "Bold & Spicy"],
    "Italian / Mediterranean":  ["Herby & Mediterranean", "Bright & Tangy"],
    "Middle Eastern":           ["Warm & Sweet", "Bright & Tangy", "Herby & Mediterranean"],
    "South Asian / Indian":     ["South Asian & Aromatic", "Warm & Sweet"],
    "American BBQ / Southern":  ["Smoky & Savory", "Bold & Spicy"],
    "French / European":        ["Herby & Mediterranean"],
    "East Asian":               ["Warm & Sweet", "Bold & Spicy"],
    "Baking & Desserts":        ["Warm & Sweet"],
}

print(f"Vocabulary: {len(CANONICAL_SPICES)} canonical spices loaded.")

Vocabulary: 34 canonical spices loaded.


In [33]:
# ── Text utilities ────────────────────────────────────────────────────────────

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def canonicalize(spice: str) -> str:
    """Normalize and apply alias mapping."""
    n = normalize_text(spice)
    return normalize_text(ALIASES.get(n, n))


def validate_spices(raw_list: list, label: str) -> set:
    """Validate spices against hardlocked vocabulary. Warns on invalid entries."""
    valid, invalid = set(), []
    for s in raw_list:
        c = canonicalize(s)
        if c in CANONICAL_SPICES:
            valid.add(c)
        else:
            invalid.append(s)
    if invalid:
        print(f"⚠️  {label} — unrecognized (ignored): {invalid}")
    return valid


# ── Spice extraction from recipe ingredients ──────────────────────────────────

SPICE_PATTERNS = sorted(
    [(sp, normalize_text(sp), re.compile(rf"(^| ){re.escape(normalize_text(sp))}( |$)"))
     for sp in SPICES],
    key=lambda x: -len(x[0])
)


def extract_spices(ingredients) -> set:
    raw = " ".join(str(x) for x in ingredients) if isinstance(ingredients, list) else str(ingredients)
    text = normalize_text(raw)
    found = set()
    for _, norm, pat in SPICE_PATTERNS:
        if pat.search(" " + text + " "):
            found.add(norm)
    return {canonicalize(sp) for sp in found}


def parse_ingredients(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        return ""
    s = x.strip()
    if s.startswith("[") and s.endswith("]"):
        items = re.findall(r"'([^']*)'|\"([^\"]*)\"", s)
        parsed = [a if a else b for a, b in items]
        return parsed if parsed else x
    return x


print("Utilities ready.")

Utilities ready.


In [34]:
# ── Core model functions ──────────────────────────────────────────────────────

def get_flavor_profiles(pantry: set, min_coverage: float = 0.3) -> list:
    """Return profiles where the user covers at least min_coverage of the profile's spices."""
    results = []
    for profile, spice_set in FLAVOR_PROFILES.items():
        matched = pantry & spice_set
        coverage = len(matched) / len(spice_set)
        if coverage >= min_coverage:
            results.append((profile, round(coverage, 2), sorted(matched)))
    return sorted(results, key=lambda x: -x[1])


def get_regions(profile_names: list) -> list:
    """Return culinary regions that match at least one of the user's flavor profiles."""
    results = []
    for region, profiles in REGION_PROFILES.items():
        matched = [p for p in profiles if p in profile_names]
        if matched:
            results.append((region, matched))
    return sorted(results, key=lambda x: -len(x[1]))


def recommend_recipes(pantry: set, must_use: set, top_k: int = 10, min_match: int = 2) -> pd.DataFrame:
    """Jaccard similarity recommender with must-use filter."""
    user_vec = mlb.transform([pantry])

    # Must-use mask: recipe must contain ALL must-use spices
    if must_use:
        spice_cols = list(mlb.classes_)
        must_idx = [spice_cols.index(s) for s in must_use if s in spice_cols]
        must_mask = X[:, must_idx].min(axis=1).astype(bool) if must_idx else np.ones(len(df_sp), dtype=bool)
    else:
        must_mask = np.ones(len(df_sp), dtype=bool)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sims = 1.0 - pairwise_distances(user_vec, X, metric="jaccard").flatten()

    match_counts = (X & user_vec).sum(axis=1)
    valid_idx = np.where(must_mask & (match_counts >= min_match))[0]

    if len(valid_idx) == 0:
        print("⚠️  No recipes matched must-use filter — relaxing constraint.")
        valid_idx = np.where(match_counts >= min_match)[0]

    ranked = valid_idx[np.lexsort((-match_counts[valid_idx], -sims[valid_idx]))][:top_k]

    out = df_sp.loc[ranked, ["title"]].copy()
    out["similarity"]     = sims[ranked].round(3)
    out["matched_spices"] = df_sp.loc[ranked, "spices"].apply(lambda s: sorted(s & pantry))
    out["num_matched"]    = match_counts[ranked]
    return out.sort_values(["similarity", "num_matched"], ascending=False).reset_index(drop=True)


def tag_region(recipe_spices: set) -> str:
    """Return the best culinary region for a recipe's spice set."""
    profiles = get_flavor_profiles(recipe_spices, min_coverage=0.2)
    if not profiles:
        return "Other"
    top_profile = profiles[0][0]
    for region, region_profiles in REGION_PROFILES.items():
        if top_profile in region_profiles:
            return region
    return "Other"


def recommend_next_spice(pantry: set, top_k: int = 5, min_match: int = 2, sim_threshold: float = 0.4) -> pd.DataFrame:
    """Rank spices not in pantry by how many new recipes they unlock."""
    baseline = recommend_recipes(pantry, must_use=set(), top_k=10_000, min_match=min_match)
    baseline_titles = set(baseline["title"])
    baseline_count  = len(baseline[baseline["similarity"] >= sim_threshold])

    missing = [s for s in CANONICAL_SPICES if s not in pantry]
    print(f"Baseline: {baseline_count} recipes at ≥{sim_threshold} similarity. Testing {len(missing)} spices...")

    results = []
    for spice in missing:
        expanded = pantry | {spice}
        new_recs = recommend_recipes(expanded, must_use=set(), top_k=10_000, min_match=min_match)
        new_count   = len(new_recs[new_recs["similarity"] >= sim_threshold])
        new_titles  = set(new_recs["title"]) - baseline_titles
        results.append({
            "spice":          spice,
            "newly_unlocked": new_count - baseline_count,
            "example_recipes": list(new_titles)[:3],
        })

    return pd.DataFrame(results).sort_values("newly_unlocked", ascending=False).head(top_k).reset_index(drop=True)


print("Model functions ready.")

Model functions ready.


---
##  Load Data
*Run once after setting your CSV path in Step 1.*

In [35]:
# Load and build spice matrix
df_raw = pd.read_csv(CSV_PATH)
if len(df_raw) > SAMPLE_SIZE:
    df_raw = df_raw.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

df_raw["ingredients_parsed"] = df_raw["ingredients"].apply(parse_ingredients)

df_sp = df_raw.copy()
df_sp["spices"] = df_sp["ingredients_parsed"].apply(extract_spices)

mlb = MultiLabelBinarizer(classes=sorted(CANONICAL_SPICES))
X   = mlb.fit_transform(df_sp["spices"])

print(f"✅ Loaded {len(df_sp):,} recipes | Matrix: {X.shape} | Avg spices/recipe: {X.sum(axis=1).mean():.2f}")

✅ Loaded 100,000 recipes | Matrix: (100000, 34) | Avg spices/recipe: 1.56


---
##  Results
*Run these cells any time you change Step 1.*

In [36]:
# ── Validate inputs from Step 1 ───────────────────────────────────────────────
pantry   = validate_spices(MY_PANTRY,  label="Pantry")
must_use = validate_spices(MUST_USE,   label="Must-use")

not_in_pantry = must_use - pantry
if not_in_pantry:
    print(f"⚠️  Must-use spices added to pantry automatically: {not_in_pantry}")
    pantry |= not_in_pantry

print(f"\nPantry   : {sorted(pantry)}")
print(f"Must-use : {sorted(must_use)}")

# ── Flavor profiles ───────────────────────────────────────────────────────────
profiles = get_flavor_profiles(pantry)

print("\n" + "━" * 60)
print("  YOUR FLAVOR PROFILES")
print("━" * 60)
for name, score, matched in profiles:
    bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
    print(f"  {bar}  {score*100:.0f}%  {name}")
    print(f"          {', '.join(matched)}")

# ── Culinary regions ──────────────────────────────────────────────────────────
regions = get_regions([p[0] for p in profiles])

print("\n" + "━" * 60)
print("  CULINARY REGIONS YOU CAN COOK")
print("━" * 60)
for region, matched_profiles in regions:
    print(f"  🍽  {region}  ({', '.join(matched_profiles)})")

# ── Top recipe recommendations ────────────────────────────────────────────────
recs = recommend_recipes(pantry, must_use, top_k=10, min_match=2)

print("\n" + "━" * 60)
print("  TOP RECIPES")
if must_use:
    print(f"  (Must contain: {sorted(must_use)})")
print("━" * 60)
print(recs[["title", "similarity", "matched_spices", "num_matched"]].to_string(index=False))

# ── Recipes grouped by region ─────────────────────────────────────────────────
candidates = recommend_recipes(pantry, must_use, top_k=200, min_match=2)
candidates["region"] = candidates["title"].apply(
    lambda t: tag_region(
        df_sp.loc[df_sp["title"] == t, "spices"].values[0]
        if len(df_sp.loc[df_sp["title"] == t]) > 0 else set()
    )
)

print("\n" + "━" * 60)
print("  RECIPES BY REGION")
print("━" * 60)
for region in candidates["region"].unique():
    if region == "Other":
        continue
    subset = candidates[candidates["region"] == region].head(3)
    print(f"\n🌍  {region}")
    for _, row in subset.iterrows():
        print(f"  • {row['title']}  (similarity: {row['similarity']})")

# ── Next spice to buy ─────────────────────────────────────────────────────────
print("\n" + "━" * 60)
print("  🛒  SPICE TO BUY NEXT")
print("━" * 60)
next_spice = recommend_next_spice(pantry, top_k=5, min_match=2)
for _, row in next_spice.iterrows():
    print(f"  {row['spice'].upper():20s} +{row['newly_unlocked']} new recipes")
    if row["example_recipes"]:
        print(f"  {'':20s} e.g. {', '.join(row['example_recipes'][:2])}")


Pantry   : ['black pepper', 'chili powder', 'cinnamon', 'cumin', 'garlic', 'ginger', 'oregano', 'paprika', 'salt', 'sumac']
Must-use : ['cumin', 'garlic', 'sumac']

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  YOUR FLAVOR PROFILES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ████████░░  83%  Smoky & Savory
          black pepper, chili powder, cumin, garlic, paprika
  ██████░░░░  67%  Bold & Spicy
          black pepper, chili powder, cumin, garlic

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CULINARY REGIONS YOU CAN COOK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🍽  Mexican / Tex-Mex  (Smoky & Savory, Bold & Spicy)
  🍽  American BBQ / Southern  (Smoky & Savory, Bold & Spicy)
  🍽  East Asian  (Bold & Spicy)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TOP RECIPES
  (Must contain: ['cumin', 'garlic', 'sumac'])
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                       